# Μέρος 3: Εφαρμογές σε Ταίριασμα και Κατηγοριοποίηση Εικόνων με Χρήση Τοπικών Περιγραφητών στα Σημεία Ενδιαφέροντος

## 3.1 Ταίριασμα Εικόνων υπό Περιστροφή και Αλλαγή Κλίμακας

Import and adjust previously used functions

In [1]:
import cv24_lab1_part3_utils as utils3
import cv24_lab1_part2_utils as utils2
import cv2
from matplotlib import pyplot as plt
import numpy as np

# Corner detection functions
def CornerDetect(image, sigma=2, ro=2.5, k=0.05, theta_corn=0.005):
    # Create structure tensor
    I_sigma = cv2.GaussianBlur(image, (0,0), sigma)
    I_sy, I_sx = np.gradient(I_sigma)
    J1 = cv2.GaussianBlur(I_sx*I_sx, (0,0), ro)
    J2 = cv2.GaussianBlur(I_sx*I_sy, (0,0), ro)
    J3 = cv2.GaussianBlur(I_sy*I_sy, (0,0), ro)

    # Calculate eigenvalues
    lamda_plus = (J1 + J3 + np.sqrt(np.square(J1-J3) + 4*np.square(J2)))/2
    lamda_minus = (J1 +         J3 - np.sqrt(np.square(J1-J3) + 4*np.square(J2)))/2

    # Corenerness criterion
    R = lamda_plus*lamda_minus - k*np.square(lamda_plus + lamda_minus)

    # Condition 1
    ns = np.ceil(3*sigma)*2+1
    B_sq = utils2.disk_strel(ns)
    Cond1 = ( R==cv2.dilate(R,B_sq) )

    # Condition 2
    Cond2 = (R > theta_corn*np.max(R))

    corners = np.logical_and(Cond1, Cond2)
    return corners

def HarrisDetector(image, sigma=2, ro=2.5, k=0.05, theta_corn=0.005, n=4):
    s = 1.5
    # Run CornerDetect for multiple scales
    sigmas = [s**i * sigma for i in range(n)]
    ros = [s**i * ro for i in range(n)]
    image_corners = [CornerDetect(image, sigma, ro, k, theta_corn) for sigma, ro in zip(sigmas, ros)]

    # Create Normalized LoG
    LoG = []
    for i in range(n):
        I_sigma = cv2.GaussianBlur(image, (0,0), sigmas[i])
        I_sy, I_sx = np.gradient(I_sigma)
        Lxx = np.gradient(I_sx)[1]
        Lyy = np.gradient(I_sy)[0]
        LoG.append((sigmas[i]**2)*abs(Lxx+Lyy))

    # For each scale, keep only corners that are maximum in area of 2 scales
    for i in range(n):
        for y, x in zip(*np.where(image_corners[i])):
            bool1 = (LoG[i][y, x] > LoG[i-1][y, x]) if i>0 else True #if-else ensures that index stays within range
            bool2 = (LoG[i][y, x] > LoG[i+1][y, x]) if i<len(LoG)-1 else True
            image_corners[i][y, x] = bool1 and bool2

    # Show result
    kp_data = []
    for i in range(n):
        for y, x in zip(*np.where(image_corners[i])):
            kp_data.append([x, y, sigmas[i]])

    return np.array(kp_data)

# Blob detection functions
def BlobDetect(image, sigma=2, theta_blob=0.005):
    # Hessian matrix
    I_sigma = cv2.GaussianBlur(image, (0,0), sigma)
    I_sy, I_sx = np.gradient(I_sigma)

    Lxx = np.gradient(I_sx)[1]
    Lyy = np.gradient(I_sy)[0]
    Lxy = np.gradient(I_sx)[0]

    # Blob criterion
    R = Lxx*Lyy - Lxy**2

    # Condition 1
    ns = np.ceil(3*sigma)*2+1
    B_sq = utils2.disk_strel(ns)
    Cond1 = ( R==cv2.dilate(R,B_sq) )

    # Condition 2
    Cond2 = (R > theta_blob*np.max(R))

    blobs = np.logical_and(Cond1, Cond2)
    return blobs

def HessianDetector(image, sigma=2, ro=2.5, k=0.05, theta_blob=0.005, n=4):
    s = 1.5
    
    # Run BlobDetect for multiple scales
    sigmas = [s**i * sigma for i in range(n)]
    ros = [s**i * ro for i in range(n)]
    image_blobs = [BlobDetect(image, sigma, theta_blob) for sigma, ro in zip(sigmas, ros)]

    # Create Normalized LoG
    LoG = []
    for i in range(n):
        I_sigma = cv2.GaussianBlur(image, (0,0), sigmas[i])
        I_sy, I_sx = np.gradient(I_sigma)
        Lxx = np.gradient(I_sx)[1]
        Lyy = np.gradient(I_sy)[0]
        LoG.append((sigmas[i]**2)*abs(Lxx+Lyy))

    # For each scale, keep only corners that are maximum in area of 2 scales
    for i in range(n):
        for y, x in zip(*np.where(image_blobs[i])):
            bool1 = (LoG[i][y, x] > LoG[i-1][y, x]) if i>0 else True #if-else ensures that index stays within range
            bool2 = (LoG[i][y, x] > LoG[i+1][y, x]) if i<len(LoG)-1 else True
            image_blobs[i][y, x] = bool1 and bool2

    # Show result
    kp_data = []
    for i in range(n):
        for y, x in zip(*np.where(image_blobs[i])):
            kp_data.append([x, y, sigmas[i]])

    return np.array(kp_data)

# Integral image sum function
def int_sum(height, width, integral_image, y, x):
    return integral_image[y, x] + integral_image[y+height, x+width] - integral_image[y+height, x] - integral_image[y, x+width]

def FastBlobDetect(image, sigma=2, theta_blob=0.005):
    # Calculate Integral image
    integral_image = np.cumsum(np.cumsum(image, axis=0), axis=1)
    integral_image = integral_image.astype(np.float64)
    
    # Apply box filters
    n = 2*np.ceil(3*sigma)+1

    dxx_height = int(4*np.floor(n/6)+1)
    dxx_width = int(2*np.floor(n/6)+1)
    dyy_height = int(2*np.floor(n/6)+1)
    dyy_width = int(4*np.floor(n/6)+1)
    dxy_width = int(2*np.floor(n/6)+1)
    dxy_height = int(2*np.floor(n/6)+1)

    integral_pad = np.pad(integral_image, ((dxx_height*2,0),(dyy_width*2,0)))
    image_height, image_width = np.shape(image)

    # Dxx
    Dxx = np.zeros(np.shape(image))
    for i in range(image_height):
        for j in range(image_width):
            Dxx[i,j] = int_sum(dxx_height, dxx_width, integral_pad, i, j) \
                    - 2*int_sum(dxx_height, dxx_width, integral_pad, i, j+dxx_width) \
                    + int_sum(dxx_height, dxx_width, integral_pad, i, j+dxx_width*2)
            
    # Dyy
    Dyy = np.zeros(np.shape(image))
    for i in range(image_height):
        for j in range(image_width):
            Dyy[i,j] = int_sum(dyy_height, dyy_width, integral_pad, i, j) \
                    - 2*int_sum(dyy_height, dyy_width, integral_pad, i+dyy_height, j) \
                    + int_sum(dyy_height, dyy_width, integral_pad, i+dyy_height*2, j)
                    
    # Dxy
    Dxy = np.zeros(np.shape(image))
    for i in range(image_height):
        for j in range(image_width):
            Dxy[i,j] = int_sum(dxy_height, dxy_width, integral_pad, i, j) \
                    - int_sum(dxy_height, dxy_width, integral_pad, i+dxy_height, j) \
                    - int_sum(dxy_height, dxy_width, integral_pad, i, j+dxy_width) \
                    + int_sum(dxy_height, dxy_width, integral_pad, i+dxy_height, j+dxy_width)

    # Blob criterion
    R = Dxx*Dyy - np.square(0.9*Dxy)

    # Condition 1
    ns = np.ceil(3*sigma)*2+1
    B_sq = utils2.disk_strel(ns)
    Cond1 = (R==cv2.dilate(R,B_sq))

    # Condition 2
    Cond2 = (R > theta_blob*np.max(R))

    blobs = np.logical_and(Cond1, Cond2)
    return blobs

def SURFDetector(image, sigma=2, ro=2.5, k=0.05, theta_blob=0.005, n=4):    
    s = 1.5
    
    # Run FastBlobDetect for multiple scales
    sigmas = [s**i * sigma for i in range(n)]
    ros = [s**i * ro for i in range(n)]
    image_blobs = [FastBlobDetect(image, sigma, theta_blob) for sigma, ro in zip(sigmas, ros)]

    # Create Normalized LoG
    LoG = []
    for i in range(n):
        I_sigma = cv2.GaussianBlur(image, (0,0), sigmas[i])
        I_sy, I_sx = np.gradient(I_sigma)
        Lxx = np.gradient(I_sx)[1]
        Lyy = np.gradient(I_sy)[0]
        LoG.append((sigmas[i]**2)*abs(Lxx+Lyy))

    # For each scale, keep only corners that are maximum in area of 2 scales
    for i in range(n):
        for y, x in zip(*np.where(image_blobs[i])):
            bool1 = (LoG[i][y, x] > LoG[i-1][y, x]) if i>0 else True #if-else ensures that index stays within range
            bool2 = (LoG[i][y, x] > LoG[i+1][y, x]) if i<len(LoG)-1 else True
            image_blobs[i][y, x] = bool1 and bool2

    # Show result
    kp_data = []
    for i in range(n):
        for y, x in zip(*np.where(image_blobs[i])):
            kp_data.append([x, y, sigmas[i]])

    return np.array(kp_data)

In [2]:
detect_fun = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.005)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

Avg. Scale Error for Image 1: 0.002
Avg. Theta Error for Image 1: 0.109


In [3]:
# mono-scaled Harris, SURF descriptor
detect_fun = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.005, 1)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n1. MONOSCALED HARRIS - SURF DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# mono-scaled Harris, HOG descriptor
detect_fun = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.005, 1)
desc_fun = lambda I, kp: utils3.featuresHOG(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n2. MONOSCALED HARRIS - HOG DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled Harris, SURF descriptor
detect_fun = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n3. MULTISCALED HARRIS - SURF DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled Harris, HOG descriptor
detect_fun = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresHOG(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n4. MULTISCALED HARRIS - HOG DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# mono-scaled Hessian, SURF descriptor
detect_fun = lambda I: HessianDetector(I, 2, 2.5, 0.05, 0.005, 1)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n5. MONOSCALED HESSIAN - SURF DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# mono-scaled Hessian, HOG descriptor
detect_fun = lambda I: HessianDetector(I, 2, 2.5, 0.05, 0.005, 1)
desc_fun = lambda I, kp: utils3.featuresHOG(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n6. MONOSCALED HESSIAN - HOG DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled Hessian, SURF descriptor
detect_fun = lambda I: HessianDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n7. MULTISCALED HESSIAN - SURF DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled Hessian, HOG descriptor
detect_fun = lambda I: HessianDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresHOG(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n8. MULTISCALED HESSIAN - HOG DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled SURF detector, SURF descriptor
detect_fun = lambda I: SURFDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresSURF(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n9. MULTISCALED SURF - SURF DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))

# multi-scaled SURF detector, HOG descriptor
detect_fun = lambda I: SURFDetector(I, 2, 2.5, 0.05, 0.005, 4)
desc_fun = lambda I, kp: utils3.featuresHOG(I,kp)
avg_scale_errors, avg_theta_errors = utils3.matching_evaluation(detect_fun, desc_fun)
print("\n10. MULTISCALED SURF - HOG DESCRIPTOR")
print('Avg. Scale Error for Image 1: {:.3f}'.format(avg_scale_errors[0]))
print('Avg. Theta Error for Image 1: {:.3f}'.format(avg_theta_errors[0]))


1. MONOSCALED HARRIS - SURF DESCRIPTOR
Avg. Scale Error for Image 1: 0.003
Avg. Theta Error for Image 1: 1.967

2. MONOSCALED HARRIS - HOG DESCRIPTOR
Avg. Scale Error for Image 1: 0.186
Avg. Theta Error for Image 1: 22.619

3. MULTISCALED HARRIS - SURF DESCRIPTOR
Avg. Scale Error for Image 1: 0.002
Avg. Theta Error for Image 1: 0.109

4. MULTISCALED HARRIS - HOG DESCRIPTOR
Avg. Scale Error for Image 1: 0.097
Avg. Theta Error for Image 1: 13.003

5. MONOSCALED HESSIAN - SURF DESCRIPTOR
Avg. Scale Error for Image 1: 0.027
Avg. Theta Error for Image 1: 7.759

6. MONOSCALED HESSIAN - HOG DESCRIPTOR
Avg. Scale Error for Image 1: 0.186
Avg. Theta Error for Image 1: 7.200

7. MULTISCALED HESSIAN - SURF DESCRIPTOR
Avg. Scale Error for Image 1: 0.001
Avg. Theta Error for Image 1: 0.078

8. MULTISCALED HESSIAN - HOG DESCRIPTOR
Avg. Scale Error for Image 1: 0.073
Avg. Theta Error for Image 1: 10.553

9. MULTISCALED SURF - SURF DESCRIPTOR
Avg. Scale Error for Image 1: 0.382
Avg. Theta Error for I

## 3.2 Κατηγοριοποίηση Εικόνων

Extract features from all data

In [13]:
harris_det = lambda I: HarrisDetector(I, 2, 2.5, 0.05, 0.05, 4)
hessian_det = lambda I: HessianDetector(I, 2, 2.5, 0.05, 0.005, 4)
surf_det = lambda I: SURFDetector(I, 2, 2.5, 0.05, 0.005, 4)
surf_desc = lambda I, kp: utils3.featuresSURF(I,kp)
hog_desc = lambda I, kp: utils3.featuresHOG(I,kp)


features = utils3.FeatureExtraction(harris_det, surf_desc)
features += features

Time for feature extraction: 253.512


In [10]:
data_train, label_train, data_test, label_test = utils3.createTrainTest(features, 1)

IndexError: list index out of range

In [20]:
accuracies = []
for k in range(5):
    # Split into training set and test set
    data_train, label_train, data_test, label_test = utils3.createTrainTest(features, k)

    # Perform Kmeans
    BOF_tr, BOF_ts = utils3.BagOfWords(data_train, data_test)

    # Train an svm on the training set and make predictions on the test set
    acc, preds, probas = utils3.svm(BOF_tr, label_train, BOF_ts, label_test)
    accuracies.append(acc)

print('Mean accuracy for Harris-Laplace with SURF descriptors: {:.3f}%'.format(100.0*np.mean(accuracies)))

IndexError: list index out of range